# Hate Speech and Offensive Language Detection on Social Media Using NLP

**Course:** IE7500 — Applied Natural Language Processing in Engineering
**Team:** Ana Luiza Young Pessoa, Mus Ab Wilmaz, Liang Wang

This notebook implements the pipeline described in the Milestone 1 project proposal:
comparing a **TF-IDF + Logistic Regression** baseline against a **fine-tuned DistilBERT**
model for 3-class classification of tweets into `hate speech`, `offensive language`, and
`neither`, using the Davidson et al. (2017) dataset.

**Sections:**
1. Setup and Imports
2. Data Loading
3. Exploratory Data Analysis (EDA)
4. Preprocessing
5. Train / Validation / Test Split
6. Baseline Model: TF-IDF + Logistic Regression
7. Transformer Model: Fine-tuned DistilBERT
8. Evaluation and Comparison
9. Bias and Error Analysis
10. Conclusions and Next Steps


## 1. Setup and Imports

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import re
import string
import random

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn (baseline model)
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)

# Imbalanced-learn (SMOTE for class imbalance handling)
# pip install imbalanced-learn
from imblearn.over_sampling import SMOTE

# Transformers (DistilBERT fine-tuning)
# pip install transformers datasets torch accelerate
import torch
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

pd.set_option("display.max_colwidth", 200)
sns.set_theme(style="whitegrid")


## 2. Data Loading

Dataset: [Davidson et al. (2017) — Hate Speech and Offensive Language](https://github.com/t-davidson/hate-speech-and-offensive-language),
mirrored on Hugging Face Datasets as `tdavidson/hate_speech_offensive`.

Label mapping:
- `0` → hate speech
- `1` → offensive language
- `2` → neither


In [ ]:
# Load dataset directly from Hugging Face (parquet)
df = pd.read_parquet("hf://datasets/tdavidson/hate_speech_offensive/data/train-00000-of-00001.parquet")

print(df.shape)
df.head()


In [ ]:
# Map numeric class to readable label
label_map = {0: "hate_speech", 1: "offensive_language", 2: "neither"}
df["label_name"] = df["class"].map(label_map)

df.info()


## 3. Exploratory Data Analysis (EDA)

- Class distribution
- Tweet length statistics
- Duplicate / missing value checks
- Vocabulary overlap across classes (TODO)


In [ ]:
# Class distribution
class_counts = df["label_name"].value_counts()
class_pct = df["label_name"].value_counts(normalize=True) * 100

print(class_counts)
print(class_pct.round(2))

plt.figure(figsize=(6, 4))
sns.barplot(x=class_counts.index, y=class_counts.values)
plt.title("Class Distribution")
plt.ylabel("Number of Tweets")
plt.xlabel("Class")
plt.show()


In [ ]:
# Missing values and duplicates
print("Missing values per column:")
print(df.isnull().sum())

print("\nDuplicate tweets:", df.duplicated(subset=["tweet"]).sum())


In [ ]:
# Tweet length statistics (in words)
df["tweet_length"] = df["tweet"].apply(lambda x: len(str(x).split()))

print(df.groupby("label_name")["tweet_length"].describe())

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="label_name", y="tweet_length")
plt.title("Tweet Length Distribution by Class (in words)")
plt.show()


**TODO (team):**
- Vocabulary overlap analysis across classes (e.g., top n-grams per class, Jaccard similarity)
- Word clouds per class (optional, for the report)


## 4. Preprocessing

- Lowercasing
- URL and @mention removal
- Punctuation / special character handling
- Tokenization (for baseline: handled by TfidfVectorizer; for DistilBERT: handled by tokenizer)


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)          # remove URLs
    text = re.sub(r"@\w+", " ", text)                        # remove mentions
    text = re.sub(r"#", "", text)                             # keep hashtag word, drop symbol
    text = re.sub(r"[^a-z\s']", " ", text)                    # remove non-alphabetic chars
    text = re.sub(r"\s+", " ", text).strip()                  # collapse whitespace
    return text

df["clean_tweet"] = df["tweet"].apply(clean_text)
df[["tweet", "clean_tweet"]].head()


## 5. Train / Validation / Test Split

Stratified split: 70% train, 15% validation, 15% test.


In [ ]:
X = df["clean_tweet"]
y = df["class"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)

print("Train:", X_train.shape, " Val:", X_val.shape, " Test:", X_test.shape)


## 6. Baseline Model: TF-IDF + Logistic Regression

Class imbalance handled via `class_weight="balanced"` (SMOTE is provided as an
alternative below — TODO: decide which strategy the team wants to keep).


In [ ]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3,
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print(X_train_tfidf.shape)


In [ ]:
# Option A: class_weight balancing
baseline_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=SEED,
)
baseline_model.fit(X_train_tfidf, y_train)


In [ ]:
# Option B (alternative, TODO: compare against Option A): SMOTE oversampling
# smote = SMOTE(random_state=SEED)
# X_train_res, y_train_res = smote.fit_resample(X_train_tfidf, y_train)
# baseline_model_smote = LogisticRegression(max_iter=1000, random_state=SEED)
# baseline_model_smote.fit(X_train_res, y_train_res)


In [ ]:
y_val_pred_baseline = baseline_model.predict(X_val_tfidf)

print(classification_report(y_val, y_val_pred_baseline, target_names=label_map.values()))
print("Macro F1 (validation):", f1_score(y_val, y_val_pred_baseline, average="macro"))


## 7. Transformer Model: Fine-tuned DistilBERT

**TODO (team, before running — this is compute/GPU intensive):**
- Confirm environment has GPU access
- Confirm batch size / epochs given available compute
- Decide max sequence length


In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize_batch(texts, max_length=128):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )

# TODO: wrap into a torch Dataset class for Trainer, e.g.:
#
# class TweetDataset(torch.utils.data.Dataset):
#     def __init__(self, encodings, labels):
#         self.encodings = encodings
#         self.labels = labels
#     def __getitem__(self, idx):
#         item = {k: v[idx] for k, v in self.encodings.items()}
#         item["labels"] = torch.tensor(self.labels.iloc[idx])
#         return item
#     def __len__(self):
#         return len(self.labels)


In [ ]:
# TODO: instantiate model, TrainingArguments, and Trainer once dataset wrapping is in place
#
# model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
#
# training_args = TrainingArguments(
#     output_dir="./results",
#     num_train_epochs=3,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=32,
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     logging_dir="./logs",
#     load_best_model_at_end=True,
#     metric_for_best_model="f1",
#     seed=SEED,
# )
#
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,
#     compute_metrics=compute_metrics,
# )
#
# trainer.train()


## 8. Evaluation and Comparison

Primary metric: **macro F1-score** (given class imbalance).
Also report per-class precision/recall/F1, confusion matrix, and ROC-AUC where applicable.


In [ ]:
def plot_confusion_matrix(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.show()

plot_confusion_matrix(
    y_val, y_val_pred_baseline, list(label_map.values()),
    "Baseline (TF-IDF + Logistic Regression) — Validation Confusion Matrix"
)


**TODO:** repeat confusion matrix / classification report for DistilBERT once trained,
and build a side-by-side comparison table (baseline vs. DistilBERT) on the held-out test set.


## 9. Bias and Error Analysis

**TODO (team):**
- Define demographic/identity term groups to probe for disparate misclassification
  (e.g., using an identity-term lexicon)
- Compare false positive / false negative rates across these groups
- Manually inspect a sample of borderline / misclassified predictions
  (hate speech vs. offensive language boundary — the primary source of error per
  Davidson et al., 2017)


In [ ]:
# Placeholder: inspect misclassified examples
errors = X_val[y_val != y_val_pred_baseline]
print(f"Number of misclassified validation examples: {len(errors)}")
errors.sample(10, random_state=SEED)


## 10. Conclusions and Next Steps

**TODO (team, after full results are in):**
- Summarize baseline vs. DistilBERT performance
- Summarize bias/error analysis findings
- Discuss ethical limitations and human-in-the-loop deployment recommendations
